# Ejercicio Módulo 5 - Dataset Swiss

**Inteligencia Artificial - CEIA - FIUBA**

**INSERTE AQUÍ SU NOMBRE**

Para practicar regresión, vamos a utilizar un dataset clásico llamado **Swiss**, que viene incluido en el lenguaje R. Contiene datos socioeconómicos de 47 provincias de habla francesa de Suiza, alrededor del año 1888. Cada fila es una provincia.

## Variables

- `Location`: provincia donde se midieron los datos.
- `Fertility`: medida estandarizada de fertilidad, en una escala de 0 a 100.
- `Agriculture`: porcentaje de hombres que trabajan en la agricultura.
- `Examination`: porcentaje de conscriptos que obtuvieron la nota más alta en el examen del ejército.
- `Education`: porcentaje de conscriptos con educación más allá de la escuela primaria.
- `Catholic`: porcentaje de la población que es católica (en lugar de protestante).
- `Infant.Mortality`: porcentaje de nacidos vivos que mueren antes de cumplir un año.

## ¿Qué queremos predecir?

Vamos a predecir la **fertilidad** (`Fertility`) de cada provincia a partir del resto de las variables, y a comparar distintos modelos de regresión. En el camino vamos a ver qué pasa cuando un modelo tiene demasiados atributos para la cantidad de datos, y cómo la **regularización** nos ayuda en ese caso.

---

Siguiendo el procedimiento típico de Machine Learning, vamos a leer los datos y separarlos en los datasets de entrenamiento y evaluación utilizando scikit-learn:

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("swiss.csv")

df.head()

In [ ]:
print(f"Tenemos {df.shape[0]} observaciones")

Obtenemos la variable objetivo (`Fertility`) y, por otro lado, los atributos. Quitamos `Location`, ya que solo identifica a la provincia y no es un atributo numérico:

In [ ]:
X = df.drop(columns=["Fertility", "Location"])
y = df["Fertility"]

Como tenemos pocas observaciones, separamos el dataset en un 50 % para entrenamiento y un 50 % para evaluación:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42
)

print(f"Observaciones de entrenamiento: {X_train.shape[0]}")
print(f"Observaciones de evaluación: {X_test.shape[0]}")

Para no repetir código, les dejamos dos funciones que vamos a usar en todo el ejercicio:

- `evaluate`: calcula el $R^2$ en el set de entrenamiento y las métricas MAE, MSE y $R^2$ en el set de evaluación. Imprime los resultados y devuelve las métricas de evaluación en un diccionario.
- `show_coefficients`: muestra en un gráfico de barras los coeficientes de uno o más modelos, con el nombre del atributo que multiplica cada uno.

Ambas reciben un `Pipeline` de scikit-learn ya entrenado, cuyo último paso es el modelo de regresión.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline


def evaluate(model: Pipeline, name: str) -> dict:
    """
    Evalúa un modelo ya entrenado e imprime sus métricas.

    Args:
        model (Pipeline): Pipeline entrenado, cuyo último paso es el regresor.
        name (str): Nombre del modelo, para mostrar en la salida.

    Returns:
        dict: MAE, MSE y R^2 del modelo en el set de evaluación.
    """
    r2_train = r2_score(y_train, model.predict(X_train))
    y_pred = model.predict(X_test)
    metrics = {
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred),
    }

    print(name)
    print(f"  R^2 en entrenamiento: {r2_train:.3f}")
    print(f"  MAE en evaluación:    {metrics['MAE']:.3f}")
    print(f"  MSE en evaluación:    {metrics['MSE']:.3f}")
    print(f"  R^2 en evaluación:    {metrics['R2']:.3f}")
    return metrics


def show_coefficients(models: dict) -> None:
    """
    Grafica los coeficientes de uno o más modelos, uno al lado del otro.

    Args:
        models (dict): Diccionario {nombre: pipeline entrenado}. Todos los
            pipelines tienen que usar los mismos atributos.
    """
    fig, axes = plt.subplots(
        1, len(models), figsize=(4 * len(models), 6), sharey=True, squeeze=False
    )
    for ax, (name, model) in zip(axes[0], models.items()):
        # Los nombres de los atributos los da el paso anterior al regresor
        names = model[:-1].get_feature_names_out()
        coefs = model[-1].coef_
        colors = ["#ff48fd" if c != 0 else "#bbbbbb" for c in coefs]
        ax.barh(names, coefs, color=colors)
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_title(name)
        ax.set_xlabel("Coeficiente")
    axes[0][0].invert_yaxis()
    plt.tight_layout()
    plt.show()

## Parte 1: Regresión lineal múltiple

Arranquemos con un modelo de **regresión lineal múltiple** que use los 5 atributos. Para ello deben:

1. Armar un `Pipeline` que escale los atributos con `StandardScaler` y luego aplique `LinearRegression`.
2. Entrenar el modelo con el set de entrenamiento.
3. Evaluarlo con la función `evaluate`, y guardar las métricas que devuelve en la variable `metrics_linear`.

In [ ]:
from sklearn.linear_model import LinearRegression  # noqa: F401
from sklearn.preprocessing import StandardScaler  # noqa: F401

##### COMPLETAR AQUÍ LO PEDIDO

## Parte 2: Agregando atributos polinómicos

Como vimos en la notebook de regresión con funciones base, podemos transformar los atributos para que la regresión lineal capture relaciones no lineales. Vamos a usar `PolynomialFeatures` de grado 2, que agrega el cuadrado de cada atributo y el producto entre cada par de atributos.

Veamos cuántos atributos quedan:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
poly.fit(X_train)

print(f"Pasamos de {X_train.shape[1]} a {len(poly.get_feature_names_out())} atributos:")
print(poly.get_feature_names_out())

Pasamos de 5 a **20 atributos**, y tenemos solo 23 observaciones de entrenamiento. Veamos qué le pasa a la regresión lineal en este caso. Para ello deben:

1. Armar un `Pipeline` que aplique `PolynomialFeatures(degree=2, include_bias=False)`, luego `StandardScaler` y, por último, `LinearRegression`.
2. Entrenar el modelo con el set de entrenamiento.
3. Evaluarlo con la función `evaluate`, y guardar las métricas en la variable `metrics_poly`.

In [ ]:
##### COMPLETAR AQUÍ LO PEDIDO

**Pregunta 1:** Comparen el $R^2$ en entrenamiento y en evaluación de este modelo. ¿Qué está pasando? ¿Por qué ocurre, teniendo en cuenta la cantidad de atributos y de observaciones de entrenamiento?

ESCRIBA AQUÍ SU RESPUESTA

## Parte 3: Regularización

La regularización nos permite usar los 20 atributos sin que el modelo sobreajuste, penalizando el tamaño de los coeficientes. Vamos a probar las dos variantes que vimos:

- **Ridge**, que penaliza la suma de los cuadrados de los coeficientes (regularización L2).
- **Lasso**, que penaliza la suma del valor absoluto de los coeficientes (regularización L1).

Para elegir el hiperparámetro $\alpha$ vamos a usar `RidgeCV` y `LassoCV`. Ambos prueban todos los valores de $\alpha$ que les pasemos con validación cruzada, eligen el que obtiene el menor error cuadrático medio promedio y, por último, entrenan el modelo final con todo el set de entrenamiento usando ese valor. El $\alpha$ elegido queda guardado en el atributo `alpha_` del modelo.

Para cada uno de los dos modelos deben:

1. Armar un `Pipeline` con los mismos pasos que en la Parte 2, pero usando `RidgeCV` (o `LassoCV`) en lugar de `LinearRegression`. Usen los valores de $\alpha$ del array `alphas` y validación cruzada de 3 folds (`cv=3`). En `LassoCV`, usen además `max_iter=100_000` para que el algoritmo llegue a converger.
2. Entrenar el modelo con el set de entrenamiento.
3. Imprimir el valor de $\alpha$ elegido.
4. Evaluarlo con la función `evaluate`, y guardar las métricas en las variables `metrics_ridge` y `metrics_lasso`.

In [ ]:
from sklearn.linear_model import LassoCV, RidgeCV  # noqa: F401

alphas = np.logspace(-3, 3, 200)

##### COMPLETAR AQUÍ LO PEDIDO

### Los coeficientes

Comparemos los coeficientes de los tres modelos con 20 atributos. Para ello, usen la función `show_coefficients`, pasándole un diccionario con los tres pipelines: la regresión lineal de la Parte 2, Ridge y Lasso. En el gráfico, los atributos cuyo coeficiente vale exactamente cero aparecen sin barra.

> ⚠️ **Nota:** Fíjense en la escala del eje horizontal de cada gráfico.

In [ ]:
##### COMPLETAR AQUÍ LO PEDIDO

Por último, impriman la lista de los atributos que **conservó** Lasso, es decir, aquellos cuyo coeficiente es distinto de cero.

> 💡 **Ayuda:** Los nombres de los atributos los pueden obtener con `get_feature_names_out()` del paso de `PolynomialFeatures`, y los coeficientes, con el atributo `coef_` del modelo.

In [ ]:
##### COMPLETAR AQUÍ LO PEDIDO

**Pregunta 2:** ¿Qué diferencia observan entre los coeficientes de Ridge y los de Lasso? ¿Qué atributos conservó Lasso?

ESCRIBA AQUÍ SU RESPUESTA

## Parte 4: Comparación de modelos y conclusiones

Completen la siguiente tabla con las métricas en el set de evaluación de cada uno de los modelos que entrenaron:

| Modelo | Atributos | MAE | MSE | $R^2$ |
| :--- | :---: | :---: | :---: | :---: |
| Regresión lineal | 5 | | | |
| Regresión lineal | 20 | | | |
| Ridge | 20 | | | |
| Lasso | 20 | | | |

**Pregunta 3:** ¿Qué modelo elegirían para predecir la fertilidad de una provincia? Justifiquen su respuesta teniendo en cuenta las métricas y también la complejidad de cada modelo (por ejemplo, cuántos atributos usa).

ESCRIBA AQUÍ SU RESPUESTA